

# Word embeddings

Francisco Jurado

<francisco.jurado@uam.es>

---


## Trabajando con Word2Vec en Python: NLTK + GenSim

_Disclaimer_: some information and examples of this notebook come from [Documentación de GenSim en NLTK](https://www.nltk.org/howto/gensim.html)

Antes de comenzar a trabajar, debemos configurar el entorno:
- NLTK
- GenSim: permite trabajar con embeddings de palabras empleando Word2Vec, FastText y Doc2Vec.

Si no lo tenemos ya instalado, instalemos NLTK:

Ahora instalemos GenSim para poder trabajar con Word2Vec:

In [ ]:
#!pip install --quiet gensim

^C


  error: subprocess-exited-with-error
  
  × Building wheel for gensim (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [716 lines of output]
      C:\Users\maxim\AppData\Local\Temp\pip-build-env-2bi4dzi9\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      C:\Users\maxim\AppData\Local\Temp\pip-build-env-2bi4dzi9\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-314\gensim
      copying gensim\downloader.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\interfaces.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\matutils.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\nosy.py -> build\lib.win-amd64-cpython-314\gensim
   

In [ ]:
from nltk.test.gensim_fixt import setup_module
setup_module() # Puede que antes tengamos que ejecutar: !pip install pytest

Si no está ya instalado, descarguemos el corpus Brown de entre los disponibles en NLTK. Este es un corpus en inglés creado en 1961 en la Universidad de Brown. Cuenta con un millón de palabras con texto de 500 fuentes (noticias, editorial, etc.)

In [ ]:
import nltk
nltk.download('brown')

Para probar, nos bastará con tomar 10000 sentencias del corpus.

In [ ]:
from nltk.corpus import brown
train_set = brown.sents()[:100000]

Entrenamos el modelo empleando el Word2Vec de GenSim:

In [ ]:
import gensim
model = gensim.models.Word2Vec(train_set)

Si se usa un corpus muy grande, puede llevar mucho tiempo entrenarlo. Lo más cómodo en esos casos es guardarlo y volver a cargarlo:

veamos cuántas palabras contiene el modelo:

In [ ]:
len(model.wv) # == model.corpus_total_words

¿Cuáles son las 100 palabras más frecuentes?

In [ ]:
model.wv.index_to_key[:100]

Una vez entrenado, el modelo contendrá la lista de palabras en el vocabulario del corpus procesado, junto con el embedding correspondiente para cada palabra.

Puede obtenese el embedding de una palabra accediendo al diccionario `wv` (word vector) que tiene el modelo.

In [ ]:
model.wv['university']

¿Cuál es la dimensión del *embedding*?

In [ ]:
len(model.wv['university'])

Para calcular la similitud entre palabras, podemos empleae la distancia coseno entre sus representaciones vectoriales?

In [ ]:
model.wv.similarity('university','school')

## Usando un modelo preentrenado.

En lugar de entrenar nuestros modelos, podemos descargar modelos de libre disposición, o emplear uno de los que tiene nltk para usar como ejemplos en `word2vec_sample`:

In [ ]:
nltk.download('word2vec_sample')

Ahora podemos cargarlo:

In [ ]:
from nltk.data import find

# fichero con los vectores en formato texto
word2vec_sample = str(find('models/word2vec_sample/pruned.word2vec.txt'))
model = gensim.models.KeyedVectors.load_word2vec_format(word2vec_sample, binary=False)

Veamos cuáles son las 3 palabras más cercanas a 'university':

In [ ]:
model.most_similar(positive=['university'], topn = 10)

Juguemos un poco...

¿cuál es la palabra que no encaja en una lista dada?

In [ ]:
model.doesnt_match(['breakfast', 'cereal', 'dinner', 'lunch'])

Veamos cómo hacer similitud semántica de género.

¿Qué vector sale si hacemos: `'woman'+'king'-'man'`?

In [ ]:
model.most_similar(positive=['woman','king'], negative=['man'], topn = 1)

Ojo, lo que nos devuelve es el vector más próximo. Si, por ejemplo, queremos sacar los 3 vectores más próximos:

In [ ]:
model.most_similar(positive=['woman','king'], negative=['man'], topn = 3)

¿Y qué saldrá de la operación `mother + daughter - woman`?

In [ ]:
model.most_similar(positive=['mother', 'daughter'], negative=['woman'], topn=1)

De forma similar:

In [ ]:
model.most_similar(positive=['father', 'daughter'], negative=['man'], topn=1)

En ocasiones es posible que no se capturen bien algunas relaciones semánticas.

Esto puede deberse a que el corpus de entrenamiento no es lo suficientemente grande o no contiene suficientes ejemplos de esa relación.

A modo de ejemplo:

In [ ]:
model.most_similar(positive=['father', 'son'], negative=['man'], topn=1)

In [ ]:
model.most_similar(positive=['France', 'Paris'], negative=['England'], topn=1)

# Construcción de una matriz de embeddings

Dado un vocabulario con el que queramos trabajar, podríamos construir una de matriz de embeddings desde Word2Vec.

Para ello, crearemos una matriz donde cada fila corresponda a la representación vectorial de una palabra en nuestro vocabulario.

Luego, podríamos utilizar esta matriz como entrada para un modelo de aprendizaje automático.

In [ ]:
import numpy as np

def create_embedding_matrix(model, tokenizer_vocab):
    embedding_dim = model.vector_size
    embedding_matrix = np.zeros((len(tokenizer_vocab), embedding_dim))

    found = 0
    for i, word in enumerate(tokenizer_vocab):
        if not hasattr(model, "wv"):
            model.wv = model
        if word in model.wv:
            embedding_matrix[i] = model.wv[word]
            found += 1

    print(f"Matriz de embeddings creada: {embedding_matrix.shape}")
    print(f"Palabras del vocabulario encontradas en el modelo: {found}/{len(tokenizer_vocab)}")

    return embedding_matrix

tokenizer_vocab = ['i', 'loved', 'movie', 'terrible', 'boring', 'amazing', 'hated', 'it']

embedding_matrix = create_embedding_matrix(model, tokenizer_vocab)

## Visualizando *embeddings*

Podríamos visualizar una proyección de los vectores de palabras en un espacio de menor dimensión, pero para ello necesitaríamos reducir la dimensión de los vectores de palabras. Para conseguilo, podemos utilizar PCA o t-SNE.

Veamos un ejemplo de cómo hacerlo con t-SNE (https://lvdmaaten.github.io/tsne/):

In [ ]:
vocab = ['university', 'school', 'faculty',
		 'king', 'prince', 'queen', 'princess',
		 'breakfast', 'dinner', 'lunch']

embedding_matrix = create_embedding_matrix(model, vocab)

# Si bien no es estrictamente necesario, es habitual usar antes PCA para reducir la dimensionalidad a unos 50 componentes principales
# para que tSNE sea más eficiente (tSNE es mucho más lento que PCA con muchas dimensiones)
from sklearn.decomposition import PCA
pca = PCA(n_components=min(50, embedding_matrix.shape[0], embedding_matrix.shape[1])) # reducimos a 50 componentes
X_50 = pca.fit_transform(embedding_matrix) # aplicamos PCA a la matriz de embeddings

# Ahora podemos usar tSNE para reducir a 2 dimensiones
from sklearn.manifold import TSNE
n_samples = X_50.shape[0]
perplexity = min(30, n_samples - 1)  # debe ser < n_samples
model_tsne = TSNE(n_components=2, random_state=0, perplexity=perplexity)
Y = model_tsne.fit_transform(X_50)

# Y finalmente, mostramos las palabras en un scatter plot
import matplotlib.pyplot as plt
plt.scatter(Y[:, 0], Y[:, 1], 20)

# Añadimos las etiquetas de las palabras
for label, x, y in zip(vocab, Y[:, 0], Y[:, 1]):
	plt.annotate(label, xy=(x, y), xytext=(0, 0), textcoords='offset points', size=10)

plt.show()

Y si quisiéramos proyectar en 3d

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Reducimos a 3 dimensiones con t-SNE
model_tsne_3d = TSNE(n_components=3, random_state=0)
n_samples_3d = X_50.shape[0]
perplexity_3d = max(1, min(30, n_samples_3d - 1))  # t-SNE exige perplexity < n_samples
model_tsne_3d = TSNE(n_components=3, random_state=0, perplexity=perplexity_3d)
Y_3d = model_tsne_3d.fit_transform(X_50)

# Mostramos las palabras en un scatter plot 3D
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(Y_3d[:, 0], Y_3d[:, 1], Y_3d[:, 2], s=20)

# Añadimos las etiquetas de las palabras
for label, x, y, z in zip(vocab, Y_3d[:, 0], Y_3d[:, 1], Y_3d[:, 2]):
    ax.text(x, y, z, label)

plt.show()

# Más sobre modelos preentrenados

Gensim proporciona varios modelos preentrenados en su repositorio [Gensim-data](https://github.com/piskvorky/gensim-data),
que se pueden cargar fácilmente con `gensim.downloader.load()`.

Para ver todos los modelos disponibles en gensim, podemos usar:

In [ ]:
import gensim.downloader

list(gensim.downloader.info()['models'].keys())

Para descargar un modelo preentrenado, basta con usar su nombre en la función load de gensim.downloader.

Probemos con el modelo "glove-twitter-25", que tiene vectores de 25 dimensiones entrenados con tweets

In [ ]:
glove_vectors = gensim.downloader.load('glove-twitter-25')
glove_vectors.most_similar('twitter')

Para cargar el modelo preentrenado de Google News (300 dimensiones, 3 millones de palabras):

In [ ]:
google_news = gensim.downloader.load('word2vec-google-news-300')
google_news.most_similar(positive=['twitter'])

Prueba las similitudes que hicimos arriba, pero esta vez con el modelo Glove y con el modelo Google News.

¿Qué diferencias observas? ¿A qué crees que se deben?